# inference
- `modeling.ipynb`에서 저장한 ONNX 변환 모델을 로드합니다.
- 평가용 데이터셋을 준비합니다.
- 추론 및 결과 검증을 진행합니다.

## 1. ONNX 변환 모델 로드



In [ ]:
!pip install onnxruntime transformers torch numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 75.6 MB/s eta 0:00:00


In [11]:
import onnxruntime as ort
import numpy as np
import time
import pickle

# 1. 파일 경로 설정
onnx_path = "/content/models/mission_10_lstm_model.onnx"
dict_path = "/content/models/word2idx_fasttext.pkl"

# 2. 단어장(word2idx) 불러오기
with open(dict_path, "rb") as f:
    word2idx_fasttext = pickle.load(f)

# 3. ONNX 세션 로드
onnx_path = "./models/mission_10_lstm_model.onnx"
ort_session = ort.InferenceSession(onnx_path)


## 2. 평가용 데이터셋 준비

In [12]:
# 토크나이저 함수
def preprocess_text(text, word_dict, max_len=280):
    words = text.split()

    indices = [word_dict.get(word, 0) for word in words]

    # Padding / Truncation
    if len(indices) < max_len:
        indices = indices + [0] * (max_len - len(indices))
    else:
        indices = indices[:max_len]

    # ONNX 모델에 맞추어 Numpy 배열 형태 (1, 280)로 리턴
    return np.array([indices], dtype=np.int64)

# 테스트 데이터
# IT/과학 분야의 짧은 뉴스 기사 원문입니다.
test_text = """
NASA's James Webb Space Telescope has captured a stunning new image
of a distant galaxy, revealing details about star formation that
were previously hidden by cosmic dust.
"""


## 3. 추론 및 결과 검증

In [20]:
import numpy as np
import json

# 1. 레이블 파일 로드
with open("./models/label_names.json", "r") as f:
    target_names = json.load(f)

# 2. 결과 해석 (Post-processing)
logits = outputs[0][0]
predicted_idx = np.argmax(logits) # 가장 높은 값의 위치 (예: 14)

predicted_category_name = target_names[predicted_idx]

print(f"예측 카테고리: {predicted_category_name}")
print(f"카테고리 인덱스: {predicted_idx}")
print(f"추론 소요 시간: {end_time - start_time:.4f}초")

예측 카테고리: sci.space
카테고리 인덱스: 14
추론 소요 시간: 0.0152초
